In [12]:
import os
import pandas as pd
import numpy as np

In [13]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Change working directory to project folder
try:
    os.chdir("/content/drive/MyDrive/UAB/FDS/campus-waste-intelligence")
    print("Directory changed")
except OSError:
    print("Error: Can't change the Current Working Directory")

Mounted at /content/drive
Directory changed


In [14]:
df_original = pd.read_csv('data/food_waste_preprocessed.csv')
print(f"Original data shape: {df_original.shape}")
df_original.head()

Original data shape: (11851, 20)


,Canteen_Section,time_bin,Waste_Weight_kg,Cost_Loss,Foot_Traffic,hour,minute,weekday,month,day_of_year,quarter,is_weekend,sin_hour,cos_hour,sin_minute,cos_minute,sin_weekday,cos_weekday,sin_month,cos_month
0,A,2015-01-01 17:00:00,3.82,11.46,100.0,17,0,3,1,1,1,0,-9.659258e-01,-2.588190e-01,0.000000e+00,1.0,0.433884,-0.900969,0.5,0.866025
1,A,2015-01-03 11:30:00,6.85,10.27,84.0,11,30,5,1,3,1,1,2.588190e-01,-9.659258e-01,5.665539e-16,-1.0,-0.974928,-0.222521,0.5,0.866025
2,A,2015-01-04 12:00:00,2.08,4.15,84.0,12,0,6,1,4,1,1,1.224647e-16,-1.000000e+00,0.000000e+00,1.0,-0.781831,0.623490,0.5,0.866025
3,A,2015-01-05 11:30:00,3.61,7.22,120.0,11,30,0,1,5,1,0,2.588190e-01,-9.659258e-01,5.665539e-16,-1.0,0.000000,1.000000,0.5,0.866025
4,A,2015-01-07 06:00:00,0.69,5.48,80.0,6,0,2,1,7,1,0,1.000000e+00,6.123234e-17,0.000000e+00,1.0,0.974928,-0.222521,0.5,0.866025


## 1. Holiday Flags

In [15]:
# Create a copy and ensure time_bin is datetime
df = df_original.copy()
df['time_bin'] = pd.to_datetime(df['time_bin'])

# Using the 'holidays' package for USA public holidays (Birmingham campus).
try:
    import holidays
    years = df['time_bin'].dt.year.unique().tolist()
    us_holidays = holidays.USA(years=years)
    df['is_holiday'] = df['time_bin'].dt.date.apply(
        lambda d: 1 if d in us_holidays else 0
    )
except ImportError:
    # Fallback: flag is_holiday as 0 if package not available
    print("'holidays' package not found – setting is_holiday=0. Install with: pip install holidays")
    df['is_holiday'] = 0

# Composite flag: holiday OR weekend
# Note: is_weekend is already in the data from preprocessing
df['is_special_day'] = ((df['is_holiday'] == 1) | (df['is_weekend'] == 1)).astype(int)

print(f"Holiday bins flagged : {df['is_holiday'].sum()}")
print(f"Special-day bins     : {df['is_special_day'].sum()}")
df[['time_bin', 'is_weekend', 'is_holiday', 'is_special_day']].drop_duplicates('time_bin').head(10)

Holiday bins flagged : 388
Special-day bins     : 3192


,time_bin,is_weekend,is_holiday,is_special_day
0,2015-01-01 17:00:00,0,1,1
1,2015-01-03 11:30:00,1,0,1
2,2015-01-04 12:00:00,1,0,1
3,2015-01-05 11:30:00,0,0,0
4,2015-01-07 06:00:00,0,0,0
5,2015-01-09 12:30:00,0,0,0
6,2015-01-10 08:00:00,1,0,1
7,2015-01-11 08:00:00,1,0,1
8,2015-01-14 17:00:00,0,0,0
9,2015-01-16 07:00:00,0,0,0


## 2. Final Dataset

In [16]:
# Select columns we need for modeling
columns_to_keep = [
    'Canteen_Section',
    'time_bin',
    'Waste_Weight_kg',
    'Cost_Loss',
    'Foot_Traffic',
    'is_holiday',
    'is_special_day'
]

df_ml = df[columns_to_keep].copy()

print(f"DataFrame shape: {df_ml.shape}")
print("Remaining columns:", df_ml.columns.tolist())
df_ml.head()

DataFrame shape: (11851, 7)
Remaining columns: ['Canteen_Section', 'time_bin', 'Waste_Weight_kg', 'Cost_Loss', 'Foot_Traffic', 'is_holiday', 'is_special_day']


,Canteen_Section,time_bin,Waste_Weight_kg,Cost_Loss,Foot_Traffic,is_holiday,is_special_day
0,A,2015-01-01 17:00:00,3.82,11.46,100.0,1,1
1,A,2015-01-03 11:30:00,6.85,10.27,84.0,0,1
2,A,2015-01-04 12:00:00,2.08,4.15,84.0,0,1
3,A,2015-01-05 11:30:00,3.61,7.22,120.0,0,0
4,A,2015-01-07 06:00:00,0.69,5.48,80.0,0,0


## 3. Save Feature-Engineered Dataset

In [17]:
output_path = 'data/food_waste_features.csv'
df_ml.to_csv(output_path, index=False)
print(f"Saved to {output_path}  |  shape: {df_ml.shape}")
print(f"\nFeature columns ({len(df_ml.columns)}):\n{list(df_ml.columns)}")

Saved to data/food_waste_features.csv  |  shape: (11851, 7)

Feature columns (7):
['Canteen_Section', 'time_bin', 'Waste_Weight_kg', 'Cost_Loss', 'Foot_Traffic', 'is_holiday', 'is_special_day']
